In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

df = pd.read_csv("cardio_cleaned_backup", sep=None, engine="python")

df_raw = df.copy(deep=True)

df_clean = df_raw.copy(deep=True)

df.head()

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,bmi
0,18393,2,168.00,62.00,110,80,1,1,0,0,1,0,50.40,21.97
1,20228,1,156.00,85.00,140,90,3,1,0,0,1,1,55.40,34.93
2,18857,1,165.00,64.00,130,70,3,1,0,0,0,1,51.60,23.51
3,17623,2,169.00,82.00,150,100,1,1,0,0,1,1,48.20,28.71
4,17474,1,156.00,56.00,100,60,1,1,0,0,0,0,47.80,23.01


In [35]:
df_clean['age_years'] = (df_clean['age'] / 365.25).round(1)

print(df_clean['age_years'].describe())

count   70000.00
mean       53.30
std         6.76
min        29.60
25%        48.40
50%        53.90
75%        58.40
max        64.90
Name: age_years, dtype: float64


In [36]:
df_clean = df_clean.drop(columns=['id'])

Anomalous height and weight values are converted to NaN to preserve the rows, and will be imputed with the Median during model building.

In [37]:
df_clean.loc[(df['height'] < 100) | (df_clean['height'] > 220), 'height'] = np.nan
df_clean.loc[df['weight'] < 30, 'weight'] = np.nan

print("Missing heights after fill:", df_clean['height'].isnull().sum())
print("Missing weights after fill:", df_clean['weight'].isnull().sum())

Missing heights after fill: 30
Missing weights after fill: 7


In [38]:
invalid_bp = (
    (df_clean['ap_hi'] <= 0) |
    (df_clean['ap_hi'] > 300) |
    (df_clean['ap_lo'] <= 0) |
    (df_clean['ap_lo'] > 200) |
    (df_clean['ap_hi'] <= df_clean['ap_lo'])
)

print("Rows to review:", invalid_bp.sum())
print("Total rows:", len(df_clean))
print("Rows remaining if removed:", len(df_clean) - invalid_bp.sum())

Rows to review: 1291
Total rows: 70000
Rows remaining if removed: 68709


In [39]:
print("Systolic BP <= 0:", (df_clean['ap_hi'] <= 0).sum())
print("Systolic BP > 300:", (df_clean['ap_hi'] > 300).sum())

print("Diastolic BP <= 0:", (df_clean['ap_lo'] <= 0).sum())
print("Diastolic BP > 200:", (df_clean['ap_lo'] > 200).sum())

print(
    "Systolic BP <= Diastolic BP:",
    (df_clean['ap_hi'] <= df_clean['ap_lo']).sum()
)

Systolic BP <= 0: 7
Systolic BP > 300: 40
Diastolic BP <= 0: 22
Diastolic BP > 200: 953
Systolic BP <= Diastolic BP: 1236


In [43]:
print("Highest systolic BP:")
print(df_clean.nlargest(10, 'ap_hi')[['ap_hi', 'ap_lo', 'cardio']])

Highest systolic BP:
       ap_hi  ap_lo  cardio
40852  16020     80       1
25464  14020     80       1
25519  14020     80       1
46912  14020     90       1
47253  14020     90       0
55459  13010     80       1
55847  13010     80       0
7763   11500     90       1
51438  11020     80       1
69370   2000    100       1


In [44]:
print("\nHighest diastolic BP:")
print(df_clean.nlargest(10, 'ap_lo')[['ap_hi', 'ap_lo', 'cardio']])


Highest diastolic BP:
       ap_hi  ap_lo  cardio
43326    200  11000       1
2381     140  10000       1
23849    150  10000       1
68538    160  10000       1
43434    130   9800       0
6653     130   9100       1
12086    130   9011       1
32920    150   9011       1
15990    120   8500       0
44042    120   8200       0


In [45]:
print("\nSystolic BP <= Diastolic BP:")
print(
    df_clean.loc[
        df_clean['ap_hi'] <= df_clean['ap_lo'],
        ['ap_hi', 'ap_lo', 'cardio']
    ].head(10)
)


Systolic BP <= Diastolic BP:
     ap_hi  ap_lo  cardio
228    160   1100       1
241    160   1000       1
260    140    800       1
329    160   1000       1
345    140   1000       1
473    150   1033       1
474    120    150       0
559    200   1000       1
567     14     90       1
613    140   1000       1


In [46]:
invalid_bp = (
    (df_clean['ap_hi'] < 70) |
    (df_clean['ap_hi'] > 250) |
    (df_clean['ap_lo'] < 40) |
    (df_clean['ap_lo'] > 150) |
    (df_clean['ap_hi'] <= df_clean['ap_lo'])
)

print("Rows to remove:", invalid_bp.sum())
print("Rows remaining:", len(df_clean) - invalid_bp.sum())

Rows to remove: 1334
Rows remaining: 68666


In [47]:
df_clean = df_clean.loc[~invalid_bp].copy()

print("Shape after cleaning:", df_clean.shape)
print("Remaining invalid BP rows:",
      (
          (df_clean['ap_hi'] < 70) |
          (df_clean['ap_hi'] > 250) |
          (df_clean['ap_lo'] < 40) |
          (df_clean['ap_lo'] > 150) |
          (df_clean['ap_hi'] <= df_clean['ap_lo'])
      ).sum())

Shape after cleaning: (68666, 13)
Remaining invalid BP rows: 0


In [1]:
df_clean["bmi"] = (
    df_clean["weight"] / (df_clean["height"] / 100) ** 2
).round(2)

df_clean.to_csv(file_path, index=False)

print("Updated shape:", df_clean.shape)
print("BMI missing values:", df_clean["bmi"].isna().sum())

display(df_clean[["height", "weight", "bmi"]].head())

Updated shape: (68666, 14)
BMI missing values: 33


,height,weight,bmi
0,168.0,62.0,21.97
1,156.0,85.0,34.93
2,165.0,64.0,23.51
3,169.0,82.0,28.71
4,156.0,56.0,23.01


In [29]:
NUMERICAL_FEATURES = ["height","weight","ap_hi","ap_lo","age_years","bmi"]

outlier_results = []

for col in NUMERICAL_FEATURES:
    s = pd.to_numeric(df[col], errors="coerce").dropna()

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_results.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outliers Count": ((s < lower) | (s > upper)).sum(),
        "Minimum": s.min(),
        "Maximum": s.max()
    })

outlier_df = pd.DataFrame(outlier_results)

display(outlier_df.round(2))

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers Count,Minimum,Maximum
0,height,159.00,170.00,11.00,142.50,186.50,478,100.00,207.00
1,weight,65.00,82.00,17.00,39.50,107.50,1739,30.00,200.00
2,ap_hi,120.00,140.00,20.00,90.00,170.00,1005,70.00,240.00
3,ap_lo,80.00,90.00,10.00,65.00,105.00,3499,40.00,120.00
4,age_years,48.30,58.40,10.10,33.15,73.55,4,29.60,64.90
5,bmi,23.88,30.12,6.24,14.51,39.49,1992,10.73,152.55


In [28]:
df_clean['bmi'] = df_clean['weight'] / (df_clean['height'] / 100) ** 2

In [13]:
df_clean.loc[df_clean['height'] < 100, 'height'] = 100

print("Minimum height:", df_clean['height'].min())
print("Maximum height:", df_clean['height'].max())

Minimum height: 100.0
Maximum height: 207.0


In [14]:
df_clean['height_category'] = np.select(
    [
        df_clean['height'] < 142.5,
        df_clean['height'] > 186.5
    ],
    [
        'Shorter than usual',
        'Taller than usual'
    ],
    default='Normal height'
)


print(df_clean['height_category'].value_counts())

height_category
Normal height         68188
Taller than usual       258
Shorter than usual      220
Name: count, dtype: int64


In [16]:
df_clean['weight_category'] = np.select(
    [
        df_clean['weight'] < 39.5,
        df_clean['weight'] > 107.5
    ],
    [
        'Lower than usual',
        'Higher than usual'
    ],
    default='Normal range'
)


print(df_clean['weight_category'].value_counts())

weight_category
Normal range         66927
Higher than usual     1696
Lower than usual        43
Name: count, dtype: int64


In [19]:
df_clean['ap_hi_category'] = np.select(
    [
        df_clean['ap_hi'] < 90,
        df_clean['ap_hi'] > 170
    ],
    [
        'Lower than usual',
        'Higher than usual'
    ],
    default='Normal IQR range'
)

print(df_clean['ap_hi_category'].value_counts())

ap_hi_category
Normal IQR range     67661
Higher than usual      932
Lower than usual        73
Name: count, dtype: int64


In [21]:
df_clean['ap_lo_category'] = np.select(
    [
        df_clean['ap_lo'] < 65,
        df_clean['ap_lo'] > 120,
        df_clean['ap_lo'] > 105
    ],
    [
        'Low outlier',
        'Extreme high - review',
        'High outlier'
    ],
    default='Within IQR range'
)

print(df_clean['ap_lo_category'].value_counts())

ap_lo_category
Within IQR range         65167
Low outlier               2860
High outlier               599
Extreme high - review       40
Name: count, dtype: int64


In [22]:
df_clean.loc[df_clean['ap_lo'] > 120, ['ap_hi', 'ap_lo']].sort_values('ap_lo')

,ap_hi,ap_lo
19871,133,121
50936,191,122
59356,180,125
17612,210,126
56422,166,126
15484,210,130
17229,180,130
14617,200,130
28423,190,130
27782,180,130


In [23]:
invalid_ap_lo = (
    (df_clean['ap_lo'] <= 0) |
    (df_clean['ap_lo'] >= df_clean['ap_hi'])
)

print("Invalid ap_lo records:", invalid_ap_lo.sum())
print(df_clean.loc[invalid_ap_lo, ['ap_hi', 'ap_lo']])

Invalid ap_lo records: 0
Empty DataFrame
Columns: [ap_hi, ap_lo]
Index: []


In [24]:
df_clean['ap_lo_original'] = df_clean['ap_lo'].copy()
df_clean['ap_lo'] = df_clean['ap_lo'].clip(upper=120)

In [25]:
age_outliers = df_clean[
    (df_clean['age_years'] < 33.15) |
    (df_clean['age_years'] > 73.55)
]

print(age_outliers['age_years'].sort_values())

21908   29.60
30072   29.70
6088    29.80
54846   30.00
Name: age_years, dtype: float64


In [35]:
missing_data = df_clean[
    df_clean['height'].isna() |
    df_clean['weight'].isna() |
    df_clean['bmi'].isna()
]

print("Number of affected rows:", len(missing_data))

display(
    missing_data[
        ['height', 'weight', 'bmi', 'age_years', 'gender', 'cardio']
    ]
)

Number of affected rows: 33


,height,weight,bmi,age_years,gender,cardio
224,NaN,55.00,NaN,59.70,2,0
6351,NaN,86.00,NaN,58.10,1,1
8005,NaN,170.00,NaN,48.30,2,1
12522,NaN,168.00,NaN,53.60,1,1
13014,NaN,68.00,NaN,61.50,2,0
14043,NaN,57.00,NaN,60.20,1,1
14875,NaN,68.00,NaN,43.30,1,0
16381,NaN,98.00,NaN,53.90,2,1
22106,NaN,65.00,NaN,39.80,1,0
22281,NaN,81.00,NaN,64.00,1,1


In [36]:
df_clean['height'] = df_clean['height'].fillna(
    df_clean.groupby('gender')['height'].transform('median')
)

df_clean['weight'] = df_clean['weight'].fillna(
    df_clean.groupby('gender')['weight'].transform('median')
)

print("Missing height:", df_clean['height'].isna().sum())
print("Missing weight:", df_clean['weight'].isna().sum())

Missing height: 0
Missing weight: 0


In [37]:
df_clean['bmi'] = df_clean['weight'] / (df_clean['height'] / 100) ** 2

print("Missing BMI:", df_clean['bmi'].isna().sum())

Missing BMI: 0


In [38]:
for col in [
    'height_category',
    'weight_category',
    'ap_hi_category',
    'ap_lo_category'
]:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False))


height_category:
height_category
Normal height         68188
Taller than usual       258
Shorter than usual      220
Name: count, dtype: int64

weight_category:
weight_category
Normal range         66927
Higher than usual     1696
Lower than usual        43
Name: count, dtype: int64

ap_hi_category:
ap_hi_category
Normal IQR range     67661
Higher than usual      932
Lower than usual        73
Name: count, dtype: int64

ap_lo_category:
ap_lo_category
Within IQR range         65167
Low outlier               2860
High outlier               599
Extreme high - review       40
Name: count, dtype: int64


In [42]:
print("Number of duplicate rows:", df_clean.duplicated().sum())

Number of duplicate rows: 0


In [41]:
df_clean.drop_duplicates(inplace=True)

print("Shape after removing duplicates:", df_clean.shape)
print("Remaining duplicate rows:", df_clean.duplicated().sum())

Shape after removing duplicates: (68642, 19)
Remaining duplicate rows: 0


In [39]:
df_clean.to_csv("cardio_cleaned.csv", index=False)